<a href="https://colab.research.google.com/github/kokila83/agenticai/blob/main/Ex%204_Agent_with_Memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import getpass
from openai import OpenAI

# 1. Get Azure AI Foundry API key securely
api_key = getpass.getpass("Enter your Azure AI Foundry API key: ")

# 2. Configure OpenAI client pointing to your Azure AI Foundry endpoint
client = OpenAI(
    api_key=api_key.strip(),
    base_url="https://ganapathy-0333-resource.services.ai.azure.com/openai/v1/"
)

# 3. Memory Agent Class
class MemoryAgent:
    def __init__(self, client):
        self.client = client
        self.conversation_history = []
        self.user_facts = {}

    def remember(self, key, value):
        self.user_facts[key] = value

    def chat(self, user_message):
        memory_context = str(self.user_facts)
        history = '\n'.join([f"{m['role']}: {m['content']}" for m in self.conversation_history[-5:]])

        prompt = f"""You are a helpful assistant with memory.

User facts: {memory_context}
Recent conversation:
{history}

User: {user_message}
Respond helpfully:"""

        # Fixed syntax error and updated model/endpoint setup
        response = self.client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[{"role": "user", "content": prompt}]
        )

        reply = response.choices[0].message.content.strip()

        # Update conversation history
        self.conversation_history.append({"role": "user", "content": user_message})
        self.conversation_history.append({"role": "assistant", "content": reply})

        # Basic entity extraction for name
        if "my name is" in user_message.lower():
            name = user_message.lower().split("my name is")[-1].strip().title()
            self.remember("name", name)

        return reply

# 4. Instantiate and test agent
if __name__ == "__main__":
    agent = MemoryAgent(client)
    print("User: Hi, my name is Rahul")
    print("AI:", agent.chat("Hi, my name is Rahul"))
    print("-" * 50)
    print("User: What's my name?")
    print("AI:", agent.chat("What's my name?"))

Enter your Azure AI Foundry API key: ··········
User: Hi, my name is Rahul
AI: Hello Rahul! How can I assist you today?
--------------------------------------------------
User: What's my name?
AI: Your name is Rahul. How can I assist you further?
